# Step 9-B — Bundle assembly

**Runs 100% locally.** No embeddings, no ChromaDB, no Gemini API -- pure
JSON/JSONL merging of files 9-A already produced plus existing RQ2/Step 8
outputs, per `step9_plan.md` §10a schema. Run this in Jupyter (or any
local Python) inside the project's WD; no Kaggle needed.

**Inputs (all already in `data/` or `queries/` or `results/`):**

| File | Role |
|---|---|
| `data/step9_query_scope.json` | the 14 neutral + 6 contradictory query_ids in scope |
| `data/step9_rerank_per_query.json` | baseline + both intervention paper lists, neutral only |
| `data/step9_gen_checkpoint.jsonl` | intervention answer_text / num2pid / citations, neutral only |
| `data/rq2_gen_checkpoint.jsonl` | baseline answer_text, neutral (Step 7a) |
| `data/rq2_frameworkB_generation_raw.jsonl` | baseline answer_text, contradictory (Step 7b) |
| `data/retrieval_results.json`, `data/retrieval_labels.json`, `data/sample_labels_1000.json`, `data/candidate_labels.json` | build baseline paper records for the 6 contradictory queries (not in step9_rerank_per_query.json, which is neutral-only) |
| `data/processed/qbio_papers.json` | titles |
| `data/rq2_frameworkA_per_query.json` | baseline Framework A diagnostics, neutral (reused, not recomputed) |
| `queries/sides_q101_150.json`, `data/rq2_frameworkB_context_judge.jsonl`, `data/rq2_frameworkB_answer_judge.jsonl` | Framework B block, contradictory |
| `results/ragas_faithfulness_result.json` | shared Faithfulness footer, both types |

**Output:** `data/step9_bundle.json`, matching `step9_plan.md` §10a
exactly (`interventions` keyed by method id, `framework_b` present only on
contradictory records, `{}` on neutral).

In [ ]:
# Cell 1: load every source file
import json

WD = '.'   # run this notebook from the project root

scope = json.load(open(f'{WD}/data/step9_query_scope.json'))
NEUTRAL_SCOPE = scope['neutral']
CONTRA_SCOPE = scope['contradictory']

rerank = json.load(open(f'{WD}/data/step9_rerank_per_query.json'))

retrieval_by_id = {q['query_id']: q for q in json.load(open(f'{WD}/data/retrieval_results.json'))}
corpus = json.load(open(f'{WD}/data/processed/qbio_papers.json'))
title_of = {str(p['paper_id']): str(p.get('title','')) for p in corpus}
cats_of  = {str(p['paper_id']): str(p.get('categories','')) for p in corpus}

baseline_gen_neutral = {r['query_id']: r for r in
    [json.loads(l) for l in open(f'{WD}/data/rq2_gen_checkpoint.jsonl')]}
baseline_gen_contra = {r['query_id']: r for r in
    [json.loads(l) for l in open(f'{WD}/data/rq2_frameworkB_generation_raw.jsonl')]}

intervention_gen = {}
for r in [json.loads(l) for l in open(f'{WD}/data/step9_gen_checkpoint.jsonl')]:
    intervention_gen[(r['query_id'], r['method'])] = r

frameworkA_by_qid = {r['query_id']: r for r in
    json.load(open(f'{WD}/data/rq2_frameworkA_per_query.json'))}

sides = {r['query_id']: r for r in json.load(open(f'{WD}/queries/sides_q101_150.json'))}
ctx_judge = {r['query_id']: r for r in
    [json.loads(l) for l in open(f'{WD}/data/rq2_frameworkB_context_judge.jsonl')]}
ans_judge = {r['query_id']: r for r in
    [json.loads(l) for l in open(f'{WD}/data/rq2_frameworkB_answer_judge.jsonl')]}

ragas = json.load(open(f'{WD}/results/ragas_faithfulness_result.json'))
ragas_by_qid = {r['query_id']: r for r in ragas['per_query']}
ragas_failed = {r['query_id'] for r in ragas['failed_queries']}

label_of = {}
for fn in ['sample_labels_1000.json', 'retrieval_labels.json', 'candidate_labels.json']:
    for r in json.load(open(f'{WD}/data/{fn}'))['labels']:
        label_of[str(r['paper_id'])] = {
            'coverage': r['coverage'], 'institution': r.get('institution'),
            'country': r.get('country'), 'elite_label': r.get('elite_label'),
        }

print(f'scope: {len(NEUTRAL_SCOPE)} neutral + {len(CONTRA_SCOPE)} contradictory')
print(f'label_of: {len(label_of):,} papers')
print(f'RAGAS corpus mean: {ragas["faithfulness_overall"]}')


In [ ]:
# Cell 1b: Faithfulness / Answer Relevancy converter (decisions D1 + D2)
#
# Rebuilds `ragas`, `ragas_by_qid`, `ragas_failed` from Raj's Step 8 checkpoint
# files, OVERRIDING what Cell 1 loaded from results/ragas_faithfulness_result.json.
# Rationale: D2 (merge_for_final.md) switches the Faithfulness headline to Raj's
# 150/150 run. The old file is left untouched on disk for provenance.
#
# 100% local: opens two .jsonl files, renames a field, computes means.
# No API call, no RAGAS re-run, no quota consumed.
#
# Output shape is identical to the old results file, so Cell 2's
# faithfulness_block() and Cell 5's meta block need no changes.
#
# IMPORTANT: this cell must run for the bundle to be correct. Use
# Restart & Run All.

# Both runs used the same judge; they differ only in context construction.
# Source: merge_for_final.md D2.
RAJ_JUDGE_MODEL = 'gemini-3.1-flash-lite'
RAJ_JUDGE_TEMPERATURE = 0

# Mandatory disclosure, minimum sentence from
# comparison_step8_with_step8_ragas.md section 4. Must be shown wherever
# 0.978 / 0.966 appears. Do not shorten to a bare number.
RAJ_CONTEXT_FORMAT_NOTE = (
    'Faithfulness was independently measured twice with two different context '
    'construction methods; this run\'s contexts were built as "{title}\\n{abstract}" '
    'without field labels, differing from the "Title: ...\\nAbstract: ..." format '
    'used at generation time and in an earlier internal run (which scored 0.9615, '
    '148/150, see results/ragas_faithfulness_result.json). The two runs converge '
    'in direction but not in exact value.'
)

import statistics

def _load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

RAJ_A = _load_jsonl(f'{WD}/results/rq2_frameworkA_ragas_cheap_checkpoint.jsonl')
RAJ_B = _load_jsonl(f'{WD}/results/rq2_frameworkB_ragas_faithfulness_checkpoint.jsonl')

def _qtype(qid):
    """q001-q100 neutral, q101-q150 contradictory. Fail loud on anything else."""
    if not (len(qid) == 4 and qid[0] == 'q' and qid[1:].isdigit()):
        raise ValueError(f'unexpected query_id format: {qid!r}')
    n = int(qid[1:])
    if 1 <= n <= 100:
        return 'neutral'
    if 101 <= n <= 150:
        return 'contradictory'
    raise ValueError(f'query_id outside q001-q150: {qid!r}')

# Float64 cosine similarity can overshoot 1.0 by ~1e-16. Values within this
# tolerance are clamped and recorded in CLAMPED below; anything beyond it still
# raises. Verified 2026-08-09: exactly one such value in Raj's files
# (q042 answer_relevancy = 1.0000000000000004, overshoot 4.4e-16).
UNIT_TOL = 1e-9
CLAMPED = []

def _unit(qid, field, v):
    """Reject null, NaN, and anything meaningfully outside [0,1].
    v != v catches NaN. Sub-tolerance float noise is clamped, not silently
    accepted: every clamp is appended to CLAMPED and printed."""
    if v is None:
        raise ValueError(f'{qid}: {field} is null')
    v = float(v)
    if v != v:
        raise ValueError(f'{qid}: {field} is NaN')
    if -UNIT_TOL <= v < 0.0 or 1.0 < v <= 1.0 + UNIT_TOL:
        CLAMPED.append((qid, field, v))
        v = min(1.0, max(0.0, v))
    if not (0.0 <= v <= 1.0):
        raise ValueError(f'{qid}: {field} outside [0,1] beyond tolerance: {v!r}')
    return v

per_query, seen = [], set()
for rec in RAJ_A + RAJ_B:
    qid = rec['query_id']
    if qid in seen:
        raise ValueError(f'duplicate query_id across checkpoints: {qid}')
    seen.add(qid)
    row = {'query_id': qid, 'type': _qtype(qid),
           'faithfulness_score': _unit(qid, 'faithfulness', rec['faithfulness'])}
    if 'answer_relevancy' in rec:
        row['answer_relevancy'] = _unit(qid, 'answer_relevancy', rec['answer_relevancy'])
    per_query.append(row)

per_query.sort(key=lambda r: r['query_id'])
if len(per_query) != 150:
    raise ValueError(f'expected 150 scored queries, got {len(per_query)}')

def _stats(vals):
    return {'n': len(vals), 'mean': statistics.mean(vals),
            'median': statistics.median(vals), 'min': min(vals), 'max': max(vals),
            'stdev': statistics.stdev(vals) if len(vals) > 1 else 0.0}

_f = [r['faithfulness_score'] for r in per_query]
_f_by = {t: [r['faithfulness_score'] for r in per_query if r['type'] == t]
         for t in ('neutral', 'contradictory')}
_ar = [r['answer_relevancy'] for r in per_query if 'answer_relevancy' in r]

ragas = {
    'metric': 'faithfulness',
    'judge_model': RAJ_JUDGE_MODEL,
    'judge_temperature': RAJ_JUDGE_TEMPERATURE,
    'source': 'Raj Step 8 checkpoints, converted in step9b Cell 1b',
    'source_files': ['results/rq2_frameworkA_ragas_cheap_checkpoint.jsonl',
                     'results/rq2_frameworkB_ragas_faithfulness_checkpoint.jsonl'],
    'context_format_disclosure': RAJ_CONTEXT_FORMAT_NOTE,
    'single_judge_disclosure': ('RAGAS Faithfulness has ONE judge. Only Framework B '
                                'has a genuine two-judge comparison. Do not conflate '
                                'the two in the UI.'),
    'query_scope': 'all 150 (100 neutral + 50 contradictory), 0 failures',
    'failed_queries': [],
    'faithfulness_overall': _stats(_f),
    'faithfulness_by_type': {t: _stats(v) for t, v in _f_by.items()},
    'answer_relevancy_overall': _stats(_ar),
    'answer_relevancy_scope': ('neutral only, n=100. The contradictory checkpoint '
                               'carries no answer_relevancy field. Never cite this '
                               'as a 150-query figure.'),
    'answer_relevancy_strictness': 1,
    'clamped_float_artifacts': [{'query_id': q, 'field': f, 'raw_value': v}
                                for q, f, v in CLAMPED],
    'answer_relevancy_ci95': [0.899, 0.927],
    'answer_relevancy_ci_source': ('D1 / comparison_step8_with_step8_ragas.md section 3. '
                                   'Not recomputed here; the mean printed below is the '
                                   'cross-check that this is the same run.'),
    'per_query': per_query,
}

ragas_by_qid = {r['query_id']: r for r in per_query}
ragas_failed = set()

print('--- Cell 1b converter (local only, no API) ---')
if CLAMPED:
    print(f'clamped {len(CLAMPED)} sub-tolerance float artifact(s):')
    for qid, field, v in CLAMPED:
        print(f'  {qid} {field} {v!r} -> clamped into [0,1]')
    print()
o = ragas['faithfulness_overall']
print(f"faithfulness  overall         n={o['n']:3d}  mean={o['mean']:.4f}")
for t in ('neutral', 'contradictory'):
    s = ragas['faithfulness_by_type'][t]
    print(f"faithfulness  {t:14s} n={s['n']:3d}  mean={s['mean']:.4f}")
s = ragas['answer_relevancy_overall']
print(f"answer_relev. neutral only    n={s['n']:3d}  mean={s['mean']:.4f}")
print()
print('Cross-check against locked decisions (must match, else STOP):')
print('  D2  faithfulness      neutral 0.978 / contradictory 0.966, 150/150')
print('  D1  answer relevancy  0.914, 95% CI [0.899, 0.927], n=100, strictness=1')


In [ ]:
# Cell 2: shared helper functions
def paper_record(rank, pid):
    lab = label_of.get(pid, {})
    return {
        'rank': rank, 'paper_id': pid, 'title': title_of.get(pid, ''),
        'institution': lab.get('institution'), 'country': lab.get('country'),
        'coverage': lab.get('coverage', 'not_found'), 'elite_label': lab.get('elite_label'),
    }

def amplification_stats(paper_recs, cited_pids_unique):
    """Same definition as rq2_methodology.md §2.1 / Framework A. Verified
    against data/rq2_frameworkA_per_query.json before writing this notebook
    (chat log): reproduces the file's own baseline numbers exactly."""
    found = [p for p in paper_recs if p['coverage'] == 'found']
    if not found:
        return {'context_elite_share': None, 'cited_elite_share': None, 'amplification': None}
    ctx_elite = sum(1 for p in found if p['elite_label'] == 1)
    ctx_share = ctx_elite / len(found)
    cited_found = [p for p in found if p['paper_id'] in cited_pids_unique]
    if not cited_found:
        return {'context_elite_share': round(ctx_share, 4), 'cited_elite_share': None, 'amplification': None}
    cited_elite = sum(1 for p in cited_found if p['elite_label'] == 1)
    cited_share = cited_elite / len(cited_found)
    return {'context_elite_share': round(ctx_share, 4), 'cited_elite_share': round(cited_share, 4),
            'amplification': round(cited_share - ctx_share, 4)}

def uniq_counts(paper_recs):
    found = [p for p in paper_recs if p['coverage'] == 'found']
    return len({p['institution'] for p in found}), len({p['country'] for p in found if p['country']})

def faithfulness_block(qid):
    if qid in ragas_failed:
        return {'status': 'failed', 'score': None}
    rec = ragas_by_qid.get(qid)
    return {'status': 'scored', 'score': rec['faithfulness_score']} if rec else {'status': 'not_run', 'score': None}

import math

K_CUT = 10

def relevance(pid, subcat):
    """Binary subcategory-match proxy, verbatim from Step 6 Cell 13.
    NOTE this is a COARSE proxy: it asks only whether the paper's arXiv
    categories string contains the query's target subcategory. It is not a
    graded semantic relevance judgment, which is why per-query NDCG here is
    illustrative and why 'no NDCG cost' must never be generalized from it."""
    return 1 if subcat in cats_of.get(pid, '') else 0

def ndcg_at_k(rels):
    dcg = sum(r / math.log2(i + 2) for i, r in enumerate(rels[:K_CUT]))
    ideal = sorted(rels, reverse=True)
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal[:K_CUT]))
    return dcg / idcg if idcg > 0 else 0.0

def mrr(rels):
    for i, r in enumerate(rels[:K_CUT]):
        if r:
            return 1.0 / (i + 1)
    return 0.0

def ir_quality(paper_recs, subcat):
    """Per-query NDCG@10 / MRR for one arm. n_relevant is carried too so the
    UI can distinguish 'scored 0 because nothing matched' from a missing value."""
    rels = [relevance(p['paper_id'], subcat) for p in paper_recs]
    return {'ndcg_at_10': round(ndcg_at_k(rels), 4),
            'mrr': round(mrr(rels), 4),
            'n_relevant': sum(rels)}

print('helpers ready')


In [ ]:
# Cell 3: build NEUTRAL query records (baseline + both interventions)
neutral_records = []
for qid in NEUTRAL_SCOPE:
    rec = rerank[qid]
    base_recs = rec['baseline']
    base_gen = baseline_gen_neutral[qid]
    base_diag_reused = frameworkA_by_qid.get(qid)   # reuse, don't recompute, for the baseline arm
    base_uinst, base_uctry = uniq_counts(base_recs)

    baseline_block = {
        'papers': base_recs,
        'answer_text': base_gen['answer_text'],
        'num2pid': base_gen['num2pid'],
        'cited_paper_ids': base_gen['cited_pids_unique'],
        'invalid_markers': base_gen['invalid_markers'],
        'diagnostics': {
            'n_labeled': base_diag_reused['context_labeled'] if base_diag_reused else None,
            'context_elite_share': base_diag_reused['context_elite_share'] if base_diag_reused else None,
            'cited_elite_share': base_diag_reused['cited_elite_share'] if base_diag_reused else None,
            'amplification': base_diag_reused['amplification'] if base_diag_reused else None,
            'uniq_institutions': base_uinst, 'uniq_countries': base_uctry,
            **ir_quality(base_recs, rec['subcategory']),
        },
    }

    interventions = {}
    for method in ['institution_aware_mmr', 'fair_top_k']:
        m_recs = rec[method]
        g = intervention_gen[(qid, method)]
        diag = amplification_stats(m_recs, g['cited_pids_unique'])
        m_uinst, m_uctry = uniq_counts(m_recs)
        n_changed = len({p['paper_id'] for p in m_recs} ^ {p['paper_id'] for p in base_recs})
        interventions[method] = {
            'papers': m_recs,
            'answer_text': g['answer_text'],
            'num2pid': g['num2pid'],
            'cited_paper_ids': g['cited_pids_unique'],
            'invalid_markers': g['invalid_markers'],
            'diagnostics': {**diag, 'uniq_institutions': m_uinst, 'uniq_countries': m_uctry,
                            **ir_quality(m_recs, rec['subcategory'])},
            'delta_vs_baseline': {
                'context_elite_share': (None if diag['context_elite_share'] is None or
                    baseline_block['diagnostics']['context_elite_share'] is None else
                    round(diag['context_elite_share'] - baseline_block['diagnostics']['context_elite_share'], 4)),
                'uniq_institutions': m_uinst - base_uinst,
                'uniq_countries': m_uctry - base_uctry,
                'n_papers_changed': n_changed,
                'ndcg_at_10': round(ir_quality(m_recs, rec['subcategory'])['ndcg_at_10']
                                    - ir_quality(base_recs, rec['subcategory'])['ndcg_at_10'], 4),
                'mrr': round(ir_quality(m_recs, rec['subcategory'])['mrr']
                             - ir_quality(base_recs, rec['subcategory'])['mrr'], 4),
            },
        }

    neutral_records.append({
        'query_id': qid, 'query_text': rec['query_text'], 'subcategory': rec['subcategory'],
        'type': 'neutral', 'is_anchor': (qid == 'q033'),
        'baseline': baseline_block, 'interventions': interventions,
        'faithfulness': faithfulness_block(qid),
    })

print(f'built {len(neutral_records)} neutral records')
print('sample diagnostics (q033):',
      next(r for r in neutral_records if r['query_id']=='q033')['baseline']['diagnostics'])


In [ ]:
# Cell 4: build CONTRADICTORY query records (baseline + framework_b, no interventions)
contra_records = []
for qid in CONTRA_SCOPE:
    q = retrieval_by_id[qid]
    pids = q['retrieved_paper_ids']
    base_recs = [paper_record(i+1, pid) for i, pid in enumerate(pids)]
    base_gen = baseline_gen_contra[qid]
    diag = amplification_stats(base_recs, base_gen['cited_pids_unique'])
    uinst, uctry = uniq_counts(base_recs)

    baseline_block = {
        'papers': base_recs,
        'answer_text': base_gen['answer_text'],
        'num2pid': base_gen['num2pid'],
        'cited_paper_ids': base_gen['cited_pids_unique'],
        'invalid_markers': base_gen['invalid_markers'],
        'diagnostics': {**diag, 'uniq_institutions': uinst, 'uniq_countries': uctry,
                        **ir_quality(base_recs, q['subcategory'])},
    }

    sd = sides[qid]
    cj = ctx_judge[qid]
    aj = ans_judge[qid]
    framework_b_block = {
        'side_a': sd['side_a'], 'side_b': sd['side_b'],
        'context_stances': cj['labels'],
        'retention_status': aj['retention_status'],
        'conclusion_favor': aj['conclusion_favor'],
        'favor_basis': aj['favor_basis'],
        'judge_evidence': aj['evidence'],
    }

    contra_records.append({
        'query_id': qid, 'query_text': q['query_text'], 'subcategory': q['subcategory'],
        'type': 'contradictory', 'is_anchor': False,
        'baseline': baseline_block, 'interventions': {},
        'framework_b': framework_b_block,
        'faithfulness': faithfulness_block(qid),
    })

print(f'built {len(contra_records)} contradictory records')
print('sample (q102) retention_status:',
      next(r for r in contra_records if r['query_id']=='q102')['framework_b']['retention_status'])


## Cell 5 - Assemble, validate, save

Fail-loud checks before saving, per this project's standing validation
rule: assert the record count matches the scope exactly, and assert every
neutral record has non-empty `interventions` while every contradictory
record has an empty one (the §10b design decision, not an oversight if
found empty on a neutral record -- that would be a real bug).

In [ ]:
# Cell 5: assemble meta + queries, validate, save
all_records = neutral_records + contra_records

assert len(all_records) == len(NEUTRAL_SCOPE) + len(CONTRA_SCOPE), 'record count mismatch'
for r in all_records:
    if r['type'] == 'neutral':
        assert r['interventions'], f"{r['query_id']}: neutral record has empty interventions"
        assert 'framework_b' not in r, f"{r['query_id']}: neutral record should not carry framework_b"
    else:
        assert r['interventions'] == {}, f"{r['query_id']}: contradictory record should have empty interventions"
        assert 'framework_b' in r, f"{r['query_id']}: contradictory record missing framework_b"
print('validation passed:', len(all_records), 'records')

bundle = {
    'meta': {
        'generated_at': '2026-08-03',
        'tier': 1,
        'n_queries_in_bundle': len(all_records),
        'n_queries_neutral_total': 100,
        'n_queries_contradictory_total': 50,
        'rerank_methods': [
            {'id': 'institution_aware_mmr', 'label': 'Institution-aware MMR',
             'lambda': 0.8, 'family': 'soft_penalty'},
            {'id': 'fair_top_k', 'label': 'Fair-Top-K',
             'elite_quota': 1, 'target_share': 0.144, 'family': 'hard_quota'},
        ],
        'generation_model': 'gemini-3.1-flash-lite',
        'generation_temperature': 0,
        'prompt_skeleton': 'step7a_evidence_grounded_no_balancing',
        'elite_list': 'QS World University Rankings 2026 Top 50',
        'sampling_seed': 42,
        'faithfulness_corpus_mean': ragas['faithfulness_overall'],
        'faithfulness_by_type': ragas['faithfulness_by_type'],
        'answer_sources': {
            'neutral_baseline': 'data/rq2_gen_checkpoint.jsonl',
            'contradictory_baseline': 'data/rq2_frameworkB_generation_raw.jsonl',
            'neutral_interventions': 'data/step9_gen_checkpoint.jsonl',
        },
    },
    'queries': all_records,
}

import os
os.makedirs('app/data', exist_ok=True)
with open('app/data/step9_bundle.json', 'w') as f:
    json.dump(bundle, f, indent=2)
size_kb = os.path.getsize('app/data/step9_bundle.json') / 1024
print(f'saved app/data/step9_bundle.json ({size_kb:.0f} KB)')


In [ ]:
# Cell 6: final summary
print('=== Step 9-B summary ===')
print(f'neutral records:       {len(neutral_records)}')
print(f'contradictory records: {len(contra_records)}')
print(f'total:                 {len(all_records)}')
print()
print('Check this is NOT gitignored before committing:')
print('  git check-ignore -v app/data/step9_bundle.json   # must print nothing')
print()
print('Next: Step 9-C wires app/streamlit_app.py to read this bundle instead of')
print('the hardcoded BASELINE_PAPERS / INTERVENTION_PAPERS_MOCK lists.')
